# Lab 3.2 · Remove one kind of context at a time

Day 3 · Session 2 (Which context helps) · 25 min. Muhsi reads three kinds of context on each metric:
synonyms in both languages, descriptions, and worked examples. Zeyad (data platform lead) has budget to
maintain one of them well. You remove each kind in turn, everything else fixed (the same model, the same
questions, run 1), re-score with the supplied scorer, and build the table that tells him which one to fund.

**AI-tutor.** An assistant may explain a result; the conclusion is yours and must cite the table.

**Reference state.** `data/assistant/certified_60.yaml`, the metric files in `data/metrics/` and the
stand-in's recorded replies `data/assistant/replies.jsonl` for the four configurations.

**Stand-in model.** Replies are recorded, synthetic, and generated by `_tools/make_case_files.py` from a
stated error model, so every figure below is reproducible offline. With `QAMAR_MODEL_MODE=live` and a free
OpenRouter key the same functions call a hosted model (the stretch).

**Tasks**
- [ ] 1 · The full configuration and its score (4')
- [ ] 2 · Without synonyms (5')
- [ ] 3 · Without descriptions (5')
- [ ] 4 · Without examples (5')
- [ ] 5 · The contribution table (6')
- [ ] ✱ Stretch · Ten questions on a free hosted model

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-408.git"
    if not Path("/content/ds-408").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-408"], check=True)
    os.chdir("/content/ds-408/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("qamar.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import json
import pandas as pd
import qamar as Q
import qsem as S
import muhsi as D

model = S.Model.load("data/semantic/qamar_model.yaml", Q.db(), metrics_dir="data/metrics")
METRICS = list(model.metrics.values())
QS = D.load_set("data/assistant/certified_60.yaml")
print(len(QS), "questions; mode:", D.MODE)

## 1 · The full configuration and its score (4')

`D.context(METRICS, variant)` is what Muhsi sees: certified metrics only, each with or without one kind of
context. `D.score(model, QS, variant)` asks every question, executes the reply on the layer and classifies it.

✏️ **Exercise** · Return, for each of the four variants, the prompt size in characters and how many metrics
carry synonyms, descriptions and examples. Each ablation must remove exactly one kind.

In [ ]:
def context_sizes() -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    sizes = context_sizes()
except NotImplementedError:
    sizes = pd.DataFrame(Q.reference("d3_2_sizes")).set_index("variant")
print(sizes.to_string())
Q.check(sizes.loc["no_synonyms", "synonyms"] == 0 and sizes.loc["no_synonyms", "examples"] == sizes.loc["full", "examples"]
        and sizes.loc["no_examples", "examples"] == 0 and sizes.loc["no_descriptions", "descriptions"] == 0,
        "each variant removes one kind and keeps the others")
full = D.score(model, QS, "full", metrics=METRICS)
print(D.summary(full).to_string())
print(full.outcome.value_counts().to_string())

🔍 Examples are a small share of the characters. Would you predict they matter least? Write your guess now;
you will check it in step 5.

## 2 · Without synonyms (5')

✏️ **Exercise** · Score `no_synonyms`, then return `flips`: the questions whose outcome differs from the full
configuration, with `id`, `lang`, `area`, the outcome in `full` and in the variant.

In [ ]:
def flips(base: pd.DataFrame, other: pd.DataFrame) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
no_syn = D.score(model, QS, "no_synonyms", metrics=METRICS)
try:
    f_syn = flips(full, no_syn)
except NotImplementedError:
    f_syn = pd.DataFrame(Q.reference("d3_2_flips_syn"))
print(D.summary(no_syn, ["lang"]).to_string())
print(f_syn.to_string(index=False))
Q.check(len(f_syn) > 0 and set(f_syn.lang) == {"ar"}, "without synonyms only Arabic questions change")

🔍 The English questions did not move. Why might synonyms matter for Arabic and not for English here? Look at
how the metrics are *named*.

## 3 · Without descriptions (5')

Run the scorer on `no_descriptions` and read its accuracy by area and your `flips`. Which area lost most?

In [ ]:
no_desc = D.score(model, QS, "no_descriptions", metrics=METRICS)
try:
    f_desc = flips(full, no_desc)
except NotImplementedError:
    f_desc = pd.DataFrame(Q.reference("d3_2_flips_desc"))
by_area = D.summary(no_desc, ["area"])
print(by_area.to_string())
print(f_desc.to_string(index=False))
Q.check(by_area.accuracy.idxmin() == "collections", "without descriptions, collections lose most")

🔍 Most collections flips are `wrong_plausible`, not errors. Which metric did Muhsi use instead? Open
`data/metrics/collections.yml` and say which sentence of the certified description tells them apart.

## 4 · Without examples (5')

Run the scorer on `no_examples` and read its accuracy by difficulty and your `flips`.

In [ ]:
no_ex = D.score(model, QS, "no_examples", metrics=METRICS)
try:
    f_ex = flips(full, no_ex)
except NotImplementedError:
    f_ex = pd.DataFrame(Q.reference("d3_2_flips_ex"))
print(D.summary(no_ex, ["difficulty"]).to_string())
print(f_ex[["id", "lang", "area", "difficulty", "outcome_variant", "got_variant", "want_full"]].to_string(index=False))
Q.check(len(f_ex) > len(f_syn) and f_ex.difficulty.min() >= 2, "examples matter most, and only above difficulty 1")

🔍 Pick two flips. What did Muhsi do to the fiscal year, the fiscal quarter or the Hijri month? Is the result
a figure anyone would notice on a slide?

## 5 · The contribution table (6')

✏️ **Exercise** · Return one row per kind of context (`synonyms`, `descriptions`, `examples`) with the drop in
accuracy from `full` overall, in Arabic and in English, the area that dropped most, and the increase in the
wrong-but-plausible rate. Sort by overall drop.

In [ ]:
def contribution(full: pd.DataFrame, variants: dict) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    table = contribution(full, {"synonyms": no_syn, "descriptions": no_desc, "examples": no_ex})
except NotImplementedError:
    table = pd.DataFrame(Q.reference("d3_2_contribution"))
print(table.to_string(index=False))
Q.check(list(table.context) == ["examples", "descriptions", "synonyms"], "examples, then descriptions, then synonyms")
(Q.lab_dir("D3_2") / "contribution.csv").write_text(table.to_csv(index=False, lineterminator="\n"), encoding="utf-8")

🔍 By overall drop, synonyms come last. Zeyad's managers ask mostly in Arabic. Write two sentences to him: which
kind to fund first, and which number in the table would change your answer.

## ✱ Stretch · Ten questions on a free hosted model

Set `OPENROUTER_API_KEY` (a free key) and `QAMAR_MODEL_MODE=live` before starting the kernel, then score ten
questions, full and without synonyms. Free models change and are rate limited: write down the model name and
the date next to any figure. Without a key this cell says so and does nothing.

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ Stretch not attempted")

## Checkpoint

1. Why remove one kind of context at a time, with the questions and the model fixed?

<details><summary>Answer</summary>So each drop in accuracy has one cause. Removing two kinds at once, or changing
the model between runs, would mix their effects.</details>

2. Without descriptions, accuracy drops and so does trust: why is the rise in wrong-but-plausible answers worse
   than the same rise in errors?

<details><summary>Answer</summary>An error is seen and the question re-asked; a plausible collection rate goes on a
slide. The wrong-but-plausible rate is the one that reaches decisions.</details>

**Carry forward.** `artifacts/D3_2/contribution.csv`. Lab 3.3 starts from the supplied files.